In [ ]:
!pip install -q rasterio

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import rasterio
from rasterio.warp import reproject, Resampling
import matplotlib.pyplot as plt

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

CKPT_DIR = '/content/drive/MyDrive/eurosat_checkpoints'   # same as training script
FINETUNE_CKPT = os.path.join(CKPT_DIR, 'best_model_finetuned.pth')

NUM_CLASSES = 10
class_names = ['AnnualCrop', 'Forest', 'HerbVeg', 'Highway', 'Industrial',
               'Pasture', 'PermCrop', 'Residential', 'River', 'SeaLake']

PATCH_SIZE = 64  # matches EuroSAT's native resolution (10m/pixel * 64 = 640m footprint)


In [ ]:
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),   # the model needs 224x224, NOT 64x64
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

Reload Model

In [ ]:
def load_model(checkpoint_path):
    model = models.resnet50(weights=None)  # architecture only, weights come from checkpoint
    num_features = model.fc.in_features
    model.fc = nn.Linear(num_features, NUM_CLASSES)
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    model = model.to(device)
    model.eval()
    return model


model = load_model(FINETUNE_CKPT)

Alignment

In [ ]:
def align_to_reference(src_path, reference_path, output_path):
    """
    Reprojects/resamples src_path onto the exact same grid (extent,
    resolution, CRS) as reference_path. Run this on your 2024 scene using
    your 2018 scene as the reference, so patch (i, j) in both images
    covers the exact same real-world location.
    """
    with rasterio.open(reference_path) as ref:
        ref_transform = ref.transform
        ref_crs = ref.crs
        ref_width, ref_height = ref.width, ref.height

    with rasterio.open(src_path) as src:
        kwargs = src.meta.copy()
        kwargs.update({
            'crs': ref_crs,
            'transform': ref_transform,
            'width': ref_width,
            'height': ref_height
        })
        with rasterio.open(output_path, 'w', **kwargs) as dst:
            for band_idx in range(1, src.count + 1):
                reproject(
                    source=rasterio.band(src, band_idx),
                    destination=rasterio.band(dst, band_idx),
                    src_transform=src.transform,
                    src_crs=src.crs,
                    dst_transform=ref_transform,
                    dst_crs=ref_crs,
                    resampling=Resampling.bilinear
                )
    print(f'Aligned {src_path} -> {output_path}')




In [ ]:
def load_rgb_array(tif_path, reflectance_max=3000):
    """
    Reads an RGB geotiff and rescales values into the 0-255 range the
    trained model expects (EuroSAT's source imagery is 8-bit RGB, not raw
    16-bit reflectance). Adjust reflectance_max if your scenes look too
    dark/bright after rescaling -- 3000 is a reasonable starting point
    for Sentinel-2 L2A surface reflectance over land, but bright surfaces
    (snow, sand, urban) may need a higher ceiling.
    """
    with rasterio.open(tif_path) as src:
        bands = src.read([1, 2, 3]).astype(np.float32)   # assumes band order R,G,B
        transform = src.transform
        crs = src.crs

    bands = np.clip(bands, 0, reflectance_max) / reflectance_max * 255.0
    bands = bands.astype(np.uint8)
    rgb = np.transpose(bands, (1, 2, 0))  # (H, W, 3) for PIL
    return rgb, transform, crs


In [ ]:
def make_patches(rgb_array, transform, patch_size=PATCH_SIZE):
    height, width, _ = rgb_array.shape
    patches = []
    positions = []  # (row, col) grid index
    geo_coords = []  # (lon/x, lat/y) of patch top-left, in the raster's CRS

    n_rows = height // patch_size
    n_cols = width // patch_size

    for row in range(n_rows):
        for col in range(n_cols):
            y0 = row * patch_size
            x0 = col * patch_size
            patch_arr = rgb_array[y0:y0 + patch_size, x0:x0 + patch_size, :]
            patch_img = Image.fromarray(patch_arr)

            patches.append(patch_img)
            positions.append((row, col))
            geo_x, geo_y = rasterio.transform.xy(transform, y0, x0)
            geo_coords.append((geo_x, geo_y))

    return patches, positions, geo_coords, n_rows, n_cols



In [ ]:
def classify_patches(patches, model, batch_size=64):
    all_preds = []
    with torch.no_grad():
        for i in range(0, len(patches), batch_size):
            batch = patches[i:i + batch_size]
            tensors = torch.stack([eval_transform(p) for p in batch]).to(device)
            outputs = model(tensors)
            _, predicted = torch.max(outputs, 1)
            all_preds.extend(predicted.cpu().numpy())
    return np.array(all_preds)


def build_landcover_map(tif_path, model, patch_size=PATCH_SIZE):
    rgb_array, transform, crs = load_rgb_array(tif_path)
    patches, positions, geo_coords, n_rows, n_cols = make_patches(
        rgb_array, transform, patch_size
    )
    preds = classify_patches(patches, model)

    grid = np.full((n_rows, n_cols), -1, dtype=int)
    for (row, col), pred in zip(positions, preds):
        grid[row, col] = pred

    return grid, geo_coords, crs


In [ ]:
SCENE_2018 = '/content/drive/MyDrive/sentinel_scenes/2018-10-31-00_00_2018-10-31-23_59_Sentinel-2_L2A_True_Color.tif'
SCENE_2024_RAW = '/content/drive/MyDrive/sentinel_scenes/2024-10-24-00_00_2024-10-24-23_59_Sentinel-2_L2A_True_Color.tif'
SCENE_2024_ALIGNED = '/content/drive/MyDrive/sentinel_scenes/karachi_2024_aligned.tif'

# Quick sanity check before anything else runs
for path in [SCENE_2018, SCENE_2024_RAW]:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f'Could not find {path}. Check that the file was uploaded/moved '
            f'into Drive under that exact name, and that Drive is mounted.'
        )

# Align 2024 onto the exact same grid as 2018 BEFORE classifying either,
# so grid[row, col] refers to the same real-world location in both maps.
align_to_reference(SCENE_2024_RAW, SCENE_2018, SCENE_2024_ALIGNED)

grid_2018, geo_2018, crs = build_landcover_map(SCENE_2018, model)
grid_2024, geo_2024, _ = build_landcover_map(SCENE_2024_ALIGNED, model)

assert grid_2018.shape == grid_2024.shape, "Grids must match after alignment"


In [ ]:
forest_idx = class_names.index('Forest')
flagged_classes = {'AnnualCrop', 'Pasture', 'Industrial', 'Residential'}
flagged_idxs = {class_names.index(c) for c in flagged_classes}

deforestation_mask = np.zeros_like(grid_2018, dtype=bool)
deforestation_events = []

n_rows, n_cols = grid_2018.shape
for row in range(n_rows):
    for col in range(n_cols):
        was_forest = grid_2018[row, col] == forest_idx
        became_flagged = grid_2024[row, col] in flagged_idxs
        if was_forest and became_flagged:
            deforestation_mask[row, col] = True
            geo_x, geo_y = geo_2018[row * n_cols + col]
            deforestation_events.append({
                'row': row, 'col': col,
                'x': geo_x, 'y': geo_y,
                'new_class': class_names[grid_2024[row, col]]
            })

print(f'Detected {len(deforestation_events)} deforestation events out of '
      f'{n_rows * n_cols} total patches.')


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(grid_2018, cmap='tab10', vmin=0, vmax=NUM_CLASSES - 1)
axes[0].set_title('2018 Land Cover')
axes[1].imshow(grid_2024, cmap='tab10', vmin=0, vmax=NUM_CLASSES - 1)
axes[1].set_title('2024 Land Cover')
axes[2].imshow(deforestation_mask, cmap='Reds')
axes[2].set_title(f'Flagged Change ({len(deforestation_events)} patches)')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/sentinel_scenes/change_detection_results.png')
plt.show()

# Save event list for later analysis (e.g. in a GIS tool)
import csv
events_csv = '/content/drive/MyDrive/sentinel_scenes/deforestation_events.csv'
with open(events_csv, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['row', 'col', 'x', 'y', 'new_class'])
    writer.writeheader()
    writer.writerows(deforestation_events)

print(f'Saved event list to {events_csv}')